# 🎱 Kino de la Suerte — Scraper + Análisis

Este notebook descarga el historial de resultados del **Kino de la Suerte**
(Lotería de Concepción, Chile) y genera un informe estadístico descriptivo:
frecuencia de números, patrones por día de sorteo, números atrasados, y
pares/impares + suma total.

**Correlo en Google Colab** (Entorno de ejecución → Ejecutar todo, o celda por
celda) — acá sí hay acceso a internet, a diferencia del entorno donde se
escribió este notebook.

> ⚠️ **Importante:** los selectores del HTML (`SELECTOR_NUMEROS`,
> `URL_RESULTADOS` en la celda de configuración) son un punto de partida
> razonable, **no verificado en vivo** contra loteria.cl. Corré primero la
> sección **"Paso 1 — Probar en una fecha conocida"**. Si falla, la celda te
> muestra el HTML crudo para que lo inspecciones (o me lo pegues a mí) y
> ajustemos el selector antes de lanzar la descarga completa.

> ⚠️ **Nota estadística:** cada sorteo es un evento independiente y aleatorio.
> La frecuencia histórica de un número no predice el próximo sorteo. Este
> análisis es descriptivo / de curiosidad, no un método de predicción.


## 0. Instalación e imports

In [ ]:
!pip install -q beautifulsoup4 tabulate


In [ ]:
from __future__ import annotations

import time
from collections import Counter
from dataclasses import dataclass
from datetime import date, datetime, timedelta
from pathlib import Path
from typing import Optional

import matplotlib.pyplot as plt
import pandas as pd
import requests
from bs4 import BeautifulSoup
from IPython.display import Markdown, display

print("Imports OK")


## 1. Configuración

Ajustá estos valores si al probar en el Paso 1 el parseo falla.

In [ ]:
# URL de resultados por fecha. AJUSTAR según la estructura real del sitio:
# abrí https://www.loteria.cl/ , buscá "resultados anteriores" / "histórico
# de sorteos", elegí una fecha y copiá el patrón de URL que use.
URL_RESULTADOS = "https://www.loteria.cl/resultados/kino?fecha={fecha}"

# Selector CSS de cada "bolita"/número ganador dentro de la página de
# resultados. AJUSTAR tras inspeccionar el HTML real (clic derecho sobre un
# número ganador → "Inspeccionar" en el navegador, o mirando el HTML crudo
# que imprime la celda de prueba más abajo).
SELECTOR_NUMEROS = ".resultado-kino .bolita, .numeros-ganadores li"

CANTIDAD_NUMEROS_KINO = 14
RANGO_NUMEROS_KINO = (1, 25)

# Días de sorteo: 0=lunes … 6=domingo. Kino es miércoles, viernes, domingo.
DIAS_SORTEO = {2, 4, 6}
NOMBRE_DIA = {
    0: "lunes", 1: "martes", 2: "miércoles", 3: "jueves",
    4: "viernes", 5: "sábado", 6: "domingo",
}

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (compatible; kino-scraper/1.0; "
        "uso personal de análisis histórico)"
    )
}

CSV_PATH = Path("/content/kino_historico.csv")
CHARTS_DIR = Path("/content/charts_kino")
INFORME_PATH = Path("/content/informe_kino.md")
CHARTS_DIR.mkdir(parents=True, exist_ok=True)

print("Configuración cargada.")


## 2. Funciones del scraper

In [ ]:
@dataclass
class ResultadoSorteo:
    fecha: date
    numeros: list[int]
    numero_sorteo: Optional[str] = None

    def to_row(self) -> dict:
        numeros_ordenados = sorted(self.numeros)
        fila = {
            "fecha": self.fecha.isoformat(),
            "dia_semana": NOMBRE_DIA[self.fecha.weekday()],
            "numero_sorteo": self.numero_sorteo or "",
            "numeros": ";".join(str(n) for n in numeros_ordenados),
        }
        for i, n in enumerate(numeros_ordenados, start=1):
            fila[f"n_{i}"] = n
        return fila


def descargar_html(fecha: date, sesion: requests.Session, reintentos: int = 4) -> Optional[str]:
    """Descarga el HTML de resultados para una fecha, con reintentos y
    backoff exponencial (2s, 4s, 8s, 16s) ante errores de red."""
    url = URL_RESULTADOS.format(fecha=fecha.isoformat())
    espera = 2
    for intento in range(1, reintentos + 1):
        try:
            resp = sesion.get(url, headers=HEADERS, timeout=20)
            if resp.status_code == 404:
                print(f"  ⚠️  Sin resultados publicados para {fecha} (404).")
                return None
            resp.raise_for_status()
            return resp.text
        except requests.RequestException as e:
            print(f"  ⚠️  Intento {intento}/{reintentos} falló para {fecha}: {e}")
            if intento == reintentos:
                print(f"  ❌ Se agotaron los reintentos para {fecha}.")
                return None
            time.sleep(espera)
            espera *= 2
    return None


def parsear_resultado(html: str, fecha: date) -> Optional[ResultadoSorteo]:
    """Extrae los 14 números ganadores del HTML. Ajustar SELECTOR_NUMEROS
    en la celda de configuración si esto no encuentra los 14 números."""
    soup = BeautifulSoup(html, "html.parser")
    elementos = soup.select(SELECTOR_NUMEROS)

    numeros: list[int] = []
    for el in elementos:
        texto = el.get_text(strip=True)
        if texto.isdigit():
            numeros.append(int(texto))

    if len(numeros) < CANTIDAD_NUMEROS_KINO:
        print(
            f"  ⚠️  Solo se encontraron {len(numeros)}/{CANTIDAD_NUMEROS_KINO} "
            f"números para {fecha} — revisar SELECTOR_NUMEROS."
        )
        return None

    numeros = numeros[:CANTIDAD_NUMEROS_KINO]
    lo, hi = RANGO_NUMEROS_KINO
    if not all(lo <= n <= hi for n in numeros):
        print(f"  ⚠️  Números fuera de rango ({lo}-{hi}) para {fecha}: {numeros}")
        return None

    return ResultadoSorteo(fecha=fecha, numeros=numeros)


def fechas_sorteo(desde: date, hasta: date, dias_sorteo: set[int]) -> list[date]:
    fechas = []
    actual = desde
    while actual <= hasta:
        if actual.weekday() in dias_sorteo:
            fechas.append(actual)
        actual += timedelta(days=1)
    return fechas

print("Funciones del scraper listas.")


## 3. Paso 1 — Probar en una fecha conocida

Corré esta celda primero. Elegí una fecha en la que sepas que hubo sorteo
(miércoles, viernes o domingo).

In [ ]:
fecha_prueba = date(2026, 8, 16)  # <-- cambiá esto por una fecha de sorteo conocida

sesion = requests.Session()
html_prueba = descargar_html(fecha_prueba, sesion)

if html_prueba is None:
    print("❌ No se pudo descargar el HTML. Revisá URL_RESULTADOS.")
else:
    resultado = parsear_resultado(html_prueba, fecha_prueba)
    if resultado is None:
        print("\n❌ No se pudieron extraer los 14 números.")
        print("Mostrando los primeros 3000 caracteres del HTML crudo para inspección:\n")
        print(html_prueba[:3000])
        print(
            "\n👉 Copiá este HTML (o el que imprime `print(html_prueba)` completo) "
            "y pegámelo para que ajuste SELECTOR_NUMEROS y URL_RESULTADOS."
        )
    else:
        print(f"✅ Parseo exitoso: {sorted(resultado.numeros)}")
        print(resultado.to_row())


## 4. Paso 2 — Descargar el histórico completo

Una vez que el Paso 1 funcione, ajustá el rango de fechas y corré esta celda.
Es reanudable: si Colab se desconecta, volvé a correr la misma celda y va a
saltar las fechas que ya estén en el CSV (mientras `CSV_PATH` no se haya
borrado del entorno).

In [ ]:
DESDE = date(2024, 8, 1)
HASTA = date.today()
PAUSA_SEGUNDOS = 1.5  # pausa cortés entre requests para no sobrecargar el sitio

CAMPOS_CSV = ["fecha", "dia_semana", "numero_sorteo", "numeros"] + [
    f"n_{i}" for i in range(1, CANTIDAD_NUMEROS_KINO + 1)
]

def cargar_fechas_existentes(ruta_csv: Path) -> set[str]:
    if not ruta_csv.exists():
        return set()
    return set(pd.read_csv(ruta_csv, dtype=str)["fecha"])


todas = fechas_sorteo(DESDE, HASTA, DIAS_SORTEO)
ya_procesadas = cargar_fechas_existentes(CSV_PATH)
pendientes = [f for f in todas if f.isoformat() not in ya_procesadas]

print(
    f"Rango {DESDE} → {HASTA}: {len(todas)} fechas de sorteo, "
    f"{len(ya_procesadas)} ya descargadas, {len(pendientes)} por descargar."
)

escribir_encabezado = not CSV_PATH.exists() or CSV_PATH.stat().st_size == 0
fallidas = []
sesion = requests.Session()

for i, fecha in enumerate(pendientes, start=1):
    print(f"[{i}/{len(pendientes)}] {fecha} ({NOMBRE_DIA[fecha.weekday()]})...", end=" ")
    html = descargar_html(fecha, sesion)
    if html is None:
        fallidas.append(fecha.isoformat())
        continue

    resultado = parsear_resultado(html, fecha)
    if resultado is None:
        fallidas.append(fecha.isoformat())
        continue

    fila = pd.DataFrame([resultado.to_row()])
    fila.to_csv(CSV_PATH, mode="a", header=escribir_encabezado, index=False, columns=CAMPOS_CSV)
    escribir_encabezado = False
    print(f"→ {';'.join(str(n) for n in sorted(resultado.numeros))}")

    time.sleep(PAUSA_SEGUNDOS)

print(f"\n✅ Listo. {len(pendientes) - len(fallidas)} sorteos guardados en {CSV_PATH}.")
if fallidas:
    print(f"⚠️  {len(fallidas)} fechas fallaron: {fallidas}")


### (Opcional) Descargar el CSV a tu computador

In [ ]:
from google.colab import files
files.download(str(CSV_PATH))


## 5. Análisis

In [ ]:
def cargar_datos(ruta_csv: Path) -> pd.DataFrame:
    df = pd.read_csv(ruta_csv, dtype={"fecha": str})
    df["fecha"] = pd.to_datetime(df["fecha"])

    def parsear_numeros(fila) -> list[int]:
        if isinstance(fila.get("numeros"), str) and fila["numeros"]:
            return [int(n) for n in fila["numeros"].split(";")]
        cols_n = [c for c in df.columns if c.startswith("n_")]
        return [int(fila[c]) for c in cols_n if pd.notna(fila[c])]

    df["lista_numeros"] = df.apply(parsear_numeros, axis=1)
    antes = len(df)
    df = df[df["lista_numeros"].apply(len) == 14].copy()
    if len(df) < antes:
        print(f"⚠️  Se descartaron {antes - len(df)} filas con menos de 14 números.")

    df["suma"] = df["lista_numeros"].apply(sum)
    df["n_pares"] = df["lista_numeros"].apply(lambda ns: sum(1 for n in ns if n % 2 == 0))
    df["n_impares"] = 14 - df["n_pares"]
    return df.sort_values("fecha").reset_index(drop=True)


df = cargar_datos(CSV_PATH)
print(f"{len(df)} sorteos cargados, de {df['fecha'].min().date()} a {df['fecha'].max().date()}.")
df.head()


In [ ]:
RANGO_NUMEROS = range(1, 26)
ORDEN_DIAS = ["lunes", "martes", "miércoles", "jueves", "viernes", "sábado", "domingo"]
COLOR_FREQ = plt.cm.Blues
COLOR_ATRASO = plt.cm.Oranges
COLOR_BASE = "#3B6FA0"

def estilo_ejes(ax) -> None:
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_visible(False)
    ax.grid(axis="x", color="#E3E3E3", linewidth=0.8, zorder=0)
    ax.set_axisbelow(True)
    ax.tick_params(length=0)

print("Estilo de gráficos listo.")


### 5.1 Frecuencia de números

In [ ]:
contador = Counter()
for lista in df["lista_numeros"]:
    contador.update(lista)
total_sorteos = len(df)

tabla_freq = pd.DataFrame(
    [(n, contador.get(n, 0)) for n in RANGO_NUMEROS], columns=["numero", "frecuencia"]
)
tabla_freq["porcentaje"] = (tabla_freq["frecuencia"] / total_sorteos * 100).round(1)
tabla_freq = tabla_freq.sort_values("frecuencia", ascending=False).reset_index(drop=True)
tabla_freq.index += 1

tabla_grafico = tabla_freq.sort_values("numero")
fig, ax = plt.subplots(figsize=(10, 5))
norm = plt.Normalize(tabla_grafico["frecuencia"].min(), tabla_grafico["frecuencia"].max())
colores = COLOR_FREQ(0.35 + 0.55 * norm(tabla_grafico["frecuencia"]))
ax.bar(tabla_grafico["numero"].astype(str), tabla_grafico["frecuencia"], color=colores, width=0.7)
ax.set_title("Frecuencia por número — Kino (todo el período)", loc="left", fontsize=12, fontweight="bold")
ax.set_xlabel("Número"); ax.set_ylabel("Veces que salió")
estilo_ejes(ax); ax.spines["bottom"].set_visible(False)
fig.tight_layout()
fig.savefig(CHARTS_DIR / "frecuencia_numeros.png", dpi=150)
plt.show()

print("Top 10 más frecuentes:")
display(tabla_freq.head(10))
print("Top 10 menos frecuentes:")
display(tabla_freq.tail(10).sort_values("frecuencia"))


### 5.2 Patrones por día de sorteo

In [ ]:
tabla_dia = (
    df.groupby("dia_semana")
    .agg(sorteos=("fecha", "count"), suma_promedio=("suma", "mean"), pares_promedio=("n_pares", "mean"))
    .reindex(ORDEN_DIAS)
    .dropna(how="all")
    .round(1)
)

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.bar(tabla_dia.index, tabla_dia["suma_promedio"], color=COLOR_BASE, width=0.55)
ax.set_title("Suma promedio de los 14 números, por día de sorteo", loc="left", fontsize=12, fontweight="bold")
ax.set_ylabel("Suma promedio")
estilo_ejes(ax); ax.spines["bottom"].set_visible(False)
fig.tight_layout()
fig.savefig(CHARTS_DIR / "patrones_por_dia.png", dpi=150)
plt.show()

display(tabla_dia)


### 5.3 Números atrasados

In [ ]:
ultima_aparicion_idx: dict[int, int] = {}
ultima_aparicion_fecha: dict[int, date] = {}

for idx, fila in df.iterrows():
    for n in fila["lista_numeros"]:
        ultima_aparicion_idx[n] = idx
        ultima_aparicion_fecha[n] = fila["fecha"].date()

filas = []
for n in RANGO_NUMEROS:
    if n in ultima_aparicion_idx:
        atraso = (total_sorteos - 1) - ultima_aparicion_idx[n]
        ultima_fecha = ultima_aparicion_fecha[n]
    else:
        atraso = total_sorteos
        ultima_fecha = None
    filas.append((n, atraso, ultima_fecha))

tabla_atraso = pd.DataFrame(filas, columns=["numero", "sorteos_sin_salir", "ultima_fecha"])
tabla_atraso = tabla_atraso.sort_values("sorteos_sin_salir", ascending=False).reset_index(drop=True)
tabla_atraso.index += 1

top20 = tabla_atraso.head(20).sort_values("sorteos_sin_salir")
fig, ax = plt.subplots(figsize=(8, 7))
norm = plt.Normalize(top20["sorteos_sin_salir"].min(), top20["sorteos_sin_salir"].max())
colores = COLOR_ATRASO(0.35 + 0.55 * norm(top20["sorteos_sin_salir"]))
ax.barh(top20["numero"].astype(str), top20["sorteos_sin_salir"], color=colores, height=0.6)
ax.set_title("Números más atrasados (top 20)", loc="left", fontsize=12, fontweight="bold")
ax.set_xlabel("Sorteos consecutivos sin salir"); ax.set_ylabel("Número")
estilo_ejes(ax); ax.spines["left"].set_visible(False)
fig.tight_layout()
fig.savefig(CHARTS_DIR / "numeros_atrasados.png", dpi=150)
plt.show()

print("Top 10 con más sorteos sin salir:")
display(tabla_atraso.head(10))


### 5.4 Pares/impares y suma total

In [ ]:
dist_pares = df["n_pares"].value_counts().sort_index()
stats_suma = df["suma"].describe().round(1)

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.bar(dist_pares.index.astype(str), dist_pares.values, color=COLOR_BASE, width=0.6)
ax.set_title("Distribución: cantidad de números pares por sorteo", loc="left", fontsize=12, fontweight="bold")
ax.set_xlabel("Cantidad de pares (de 14 números)"); ax.set_ylabel("Sorteos")
estilo_ejes(ax); ax.spines["bottom"].set_visible(False)
fig.tight_layout()
fig.savefig(CHARTS_DIR / "pares_impares.png", dpi=150)
plt.show()

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.hist(df["suma"], bins=20, color=COLOR_BASE, edgecolor="white")
ax.axvline(df["suma"].mean(), color="#B23B3B", linewidth=2, linestyle="--", label=f"Media ({df['suma'].mean():.0f})")
ax.set_title("Distribución de la suma de los 14 números ganadores", loc="left", fontsize=12, fontweight="bold")
ax.set_xlabel("Suma"); ax.set_ylabel("Sorteos")
ax.legend(frameon=False)
estilo_ejes(ax); ax.spines["bottom"].set_visible(False)
fig.tight_layout()
fig.savefig(CHARTS_DIR / "distribucion_suma.png", dpi=150)
plt.show()

print("Distribución de pares por sorteo:")
display(dist_pares.rename("sorteos").to_frame())
print("Estadísticas de la suma:")
display(stats_suma.rename("valor").to_frame())


## 6. Informe Markdown final

In [ ]:
def tabla_md(d: pd.DataFrame) -> str:
    return d.to_markdown()

desde_real, hasta_real = df["fecha"].min().date(), df["fecha"].max().date()

informe = f'''# Informe de análisis — Kino de la Suerte

**Período analizado:** {desde_real} a {hasta_real}
**Sorteos incluidos:** {len(df)}
**Generado:** {date.today().isoformat()}

> ⚠️ Cada sorteo es independiente y aleatorio. Este informe es descriptivo, no predictivo.

## 1. Frecuencia de números
![Frecuencia por número](charts_kino/frecuencia_numeros.png)

**Top 10 más frecuentes:**

{tabla_md(tabla_freq.head(10))}

**Top 10 menos frecuentes:**

{tabla_md(tabla_freq.tail(10).sort_values("frecuencia"))}

## 2. Patrones por día de sorteo
![Patrones por día](charts_kino/patrones_por_dia.png)

{tabla_md(tabla_dia)}

## 3. Números atrasados
![Números atrasados](charts_kino/numeros_atrasados.png)

**Top 10 con más sorteos sin salir:**

{tabla_md(tabla_atraso.head(10))}

## 4. Pares/impares y suma total
![Distribución pares/impares](charts_kino/pares_impares.png)
![Distribución de la suma](charts_kino/distribucion_suma.png)

**Distribución de cantidad de números pares por sorteo:**

{tabla_md(dist_pares.rename("sorteos").to_frame())}

**Estadísticas de la suma de los 14 números:**

{tabla_md(stats_suma.rename("valor").to_frame())}
'''

INFORME_PATH.write_text(informe, encoding="utf-8")
display(Markdown(informe))


### (Opcional) Descargar informe + gráficos

In [ ]:
import shutil
from google.colab import files

zip_path = shutil.make_archive("/content/kino_informe", "zip", root_dir="/content",
                                base_dir=".")  # incluye informe_kino.md, charts_kino/, kino_historico.csv
files.download("/content/kino_informe.zip")
